**Continuous Stirred Tank Reactor (CSTR): Methanol Carbonylation**

In this tutorial we model acetic acid production by methanol carbonylation in an adiabatic CSTR, and compare the result with an equivalent DWSIM CSTR model (UNIQUAC property package).

A CSTR uses **reaction kinetics**: the reaction rate is evaluated at the outlet conditions (the reactor contents are perfectly mixed), and the reactor volume sets how much reaction takes place:

$$F_{j,\mathrm{out}} = F_{j,\mathrm{in}} + \nu_j \, r \, V$$

### Reaction and kinetics:

$$\mathrm{CH_3OH + CO \rightarrow CH_3COOH} \qquad r = k \, C_{\mathrm{MeOH}} \, C_{\mathrm{CO}}^{0.5}, \quad k = 3.5\times10^{6} \exp\left(-\frac{83\,680}{RT}\right)$$

with r in mol/m³/s and concentrations in mol/m³.

### Feed and reactor specifications:

| | |
|---|---|
| Feed | 1 kg/s at 328 K and 1.01325 bar: methanol 20.46, water 18.47, CO 0.417 mol/s |
| Reactor volume | 100 m³ |
| Operation | adiabatic, no pressure drop |

---

> **Mixture concentrations**
>
> DWSIM evaluates the rate on the "Mixture" phase: each concentration is the component molar flow divided by the total volumetric flow of the reactor outlet. `carbonylation_reaction.py` uses a small custom rate form that does exactly this, so the rate does not depend on how the stream is split between vapor and liquid.

> **Why this comparison is approximate**
>
> The DWSIM model uses the UNIQUAC activity coefficient model, Lee-Kesler liquid enthalpies and Henry's law for carbon monoxide. The IDAES generic property framework has no UNIQUAC model, so `methanol_carbonylation.py` approximates it:
>
> * pure component data (vapor pressure, ideal gas heat capacity, liquid density) are the same ChemSep coefficients DWSIM uses, read from the DWSIM model file;
> * the liquid is ideal (Raoult's law, activity coefficients of 1);
> * the liquid enthalpy is the ideal gas enthalpy minus the heat of vaporization from the Pitzer corresponding states correlation, similar in spirit to DWSIM's Lee-Kesler method;
> * carbon monoxide, far above its critical temperature, is a non-condensable gas.
>
> The feed is two-phase: carbon monoxide and some methanol and water vapor enter as gas, and the product is (almost) all liquid. Without the liquid non-ideality and the solubility of CO, IDAES puts about 12 % more vapor in the feed, and the extra condensation in the reactor makes the outlet slightly warmer. The reaction itself is reproduced closely.

In [ ]:
import idaes

print("IDAES version: ", idaes.__version__)

## Setting up and solving the flowsheet

The flowsheet is Feed → CSTR → Product, built in `cstr.py`. The CSTR has phase equilibrium enabled, so the two-phase feed condenses as it reacts.

In [ ]:
from cstr import setup_cstr_model, report_cstr_properties

# Create and solve the model
model, results = setup_cstr_model(
    feed={
        "flow_mol_comp": {"methanol": 20.4600949, "water": 18.4691703, "acetic_acid": 0, "carbon_monoxide": 0.41746463},  # mol/s
        "temperature": 328,  # K
        "pressure": 101325,  # Pa
    },
    volume=100,  # m3
)
print(results.solver.termination_condition)

prop = report_cstr_properties(model)

In [ ]:
from pyomo.environ import value

cv = model.fs.cstr.control_volume
outlet = cv.properties_out[0]
print("Reaction rate x volume: {0:.4f} mol/s".format(value(cv.rate_reaction_extent[0, "carbonylation"])))
print("Rate constant at the outlet temperature: {0:.4e} m^1.5/mol^0.5/s".format(value(cv.reactions[0].k_rxn["carbonylation"])))
print("Outlet volumetric flow: {0:.6f} m3/s, residence time: {1:.1f} h".format(
    value(outlet.flow_vol), value(model.fs.cstr.volume[0] / outlet.flow_vol) / 3600))

In [ ]:
# Visualize (if needed)
model.fs.visualize("CSTR Model")

## DWSIM model

In [ ]:
from IPython.display import Image
Image(filename="DWSim CSTR Model.png")

In [ ]:
import webbrowser
import tempfile
import os

def display_image_in_browser(image_path, title="Image Viewer"):
    # Create a temporary HTML file that embeds the image
    html_content = f"""
    <html>
        <head>
            <title>{title}</title>
        </head>
        <body style="text-align: center;">
            <h2>{title}</h2>
            <img src="file://{os.path.abspath(image_path)}" style="max-width:90%; max-height:90%;">
        </body>
    </html>
    """
    with tempfile.NamedTemporaryFile('w', delete=False, suffix='.html') as f:
        f.write(html_content)
        temp_html_path = f.name

    webbrowser.open(f'file://{temp_html_path}')


In [ ]:
display_image_in_browser("DWSim CSTR Model.png", title="DWSim CSTR Model Check")

## Comparison with DWSIM

The DWSIM results are read directly from the solved `.dwxmz` file and compared with IDAES. The conversion and product flows agree closely. The outlet temperature is slightly higher in IDAES because of the larger feed vapor flow, and the small amount of unreacted carbon monoxide is lower because the rate constant rises with temperature (the outlet CO scales with 1/k²).

In [ ]:
from cstr import read_dwsim_results, compare_with_dwsim

dwsim = read_dwsim_results("DWSim CSTR Model.dwxmz")
compare_with_dwsim(model, dwsim)